In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import lightning as L

from src.data.dataset import MLMPretrainingDataset

In [ ]:
MAX_LENGTH = 256
BATCH_SIZE = 64 
MODEL_DIM = 2048
NUM_HEADS = 32
NUM_LAYERS = 4

MASK_PROB = 0.15
RAND_PROB = 0.1

In [ ]:
train_data_raw = torch.load(
    r"..\data\processed\human_nontata_promoters\train\nucleotide_dataset.pt"
)
test_data_raw = torch.load(
    r"..\data\processed\human_nontata_promoters\test\nucleotide_dataset.pt"
)

In [ ]:
len(train_data_raw["sequences"])

In [ ]:
train_data = MLMPretrainingDataset(
    train_data_raw,
    mask_prob=MASK_PROB,
    rand_prob=RAND_PROB,
    max_length=MAX_LENGTH,
) 

test_data = MLMPretrainingDataset(
    test_data_raw,
    mask_prob=MASK_PROB,
    rand_prob=RAND_PROB,
    max_length=MAX_LENGTH,
)

In [ ]:
train_data.__len__()

In [ ]:
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, persistent_workers=True, pin_memory=True, prefetch_factor=2)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, persistent_workers=True, pin_memory=True, prefetch_factor=2)

print(f"Batch sample: {next(iter(train_loader))[0].shape}")

In [ ]:
from src.models.transformer import Transformer
import torch.nn.functional as F

In [ ]:
encoder = Transformer(
    num_unique_tokens=9,  # 4 nucleotides + 5 special tokens
    model_dim=MODEL_DIM,
    num_heads=NUM_HEADS,
    max_len=MAX_LENGTH
)

In [ ]:
seq_input = next(iter(train_loader))
print(seq_input[0])

In [ ]:
class Model(L.LightningModule):
    def __init__(self, transformer, learning_rate=1e-4):
        super(Model, self).__init__()
        self.save_hyperparameters(ignore=['transformer'])

        self.embedder = nn.Embedding(9, MODEL_DIM)  # 9 unique tokens
        self.transformer = transformer
        self.output_projection = nn.Linear(MODEL_DIM, 9)  # Output layer for token prediction
        self.output_projection.weight = self.embedder.weight

        self.criterion = nn.CrossEntropyLoss()
        self.learning_rate = learning_rate

    def forward(self, x, mask=None):
        embedded_x = self.embedder(x)
        transformer_output = self.transformer(embedded_x, mask=mask)
        logits = self.output_projection(transformer_output)
        return logits

    def training_step(self, batch, batch_idx):
        input_ids = batch[0]
        mask = batch[1]
        labels = batch[2]

        logits = self(input_ids, mask)
        loss = self.criterion(logits.view(-1, 9), labels.view(-1))
        
        # Log metrics
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        input_ids = batch[0]
        mask = batch[1]
        labels = batch[2]

        logits = self(input_ids, mask)
        loss = self.criterion(logits.view(-1, 9), labels.view(-1))
        
        self.log('val_loss', loss, on_epoch=True, prog_bar=True)
        return loss


    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        
        # Optional: Add learning rate scheduler
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=5
        )
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_loss'
            }
        }

In [ ]:
transformer = Model(transformer=encoder)

In [ ]:
transformer_output = transformer(seq_input[0], seq_input[1])

In [ ]:
transformer

In [ ]:
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping

In [ ]:
def train_lightning(model, train_loader, val_loader, max_epochs=10):
    trainer = L.Trainer(
        max_epochs=max_epochs,
        accelerator='auto', 
        devices=1,
        log_every_n_steps=10,
        enable_checkpointing=True,
        callbacks=[
            ModelCheckpoint(  
                monitor='val_loss',
                mode='min',
                save_top_k=1,
                filename='best-model-{epoch:02d}-{val_loss:.2f}'
            ),
            EarlyStopping(
                monitor='val_loss',
                patience=3,
                mode='min'
            )
        ]
    )

    trainer.fit(model, train_loader, val_loader)
    
    return model, trainer

In [ ]:
transformer, trainer = train_lightning(transformer, train_loader, test_loader, max_epochs=100)

In [ ]:
trainer.validate(model=transformer, dataloaders=test_loader)

In [ ]:
trained_transformer_output = transformer(seq_input[0], seq_input[1])